In [1]:
import os
import pandas as pd
import psycopg2
import numpy as np
import tensorflow as tf
from io import BytesIO
import json
import tempfile

2026-02-18 03:01:27.196243: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-18 03:01:27.297343: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-18 03:01:30.757265: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-18 03:01:57.098844: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [18]:
# Connect to database
conn = psycopg2.connect(
    user="ml_admin",
    password="password",
    host="127.0.0.1",
    port="5432",
    database="ml-artifacts"
)
cursor = conn.cursor()

In [19]:
# Name of table containing model
table_name = "training_history"

In [4]:
# Show first 5 rows of the table
query = f"SELECT * FROM {table_name} LIMIT 5;"
df_head = pd.read_sql_query(query, conn)
df_head

/tmp/ipykernel_424375/2078807494.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_head = pd.read_sql_query(query, conn)


,run_id,batch_id,source_table,target_column,rows_limit,time_start,time_end,algorithm,model_object,model_weights,metrics,created_at
0,1,9b881d69-d956-4f7e-83eb-7380688adb28,iot_pond_3,temperature,5000,2021-09-15 17:00:00,2021-10-12 14:00:00,LSTM,"[b'P', b'K', b'\x03', b'\x04', b'\x14', b'\x00...","{'scaler_min': -1.0595291013947994, 'scaler_sc...","{'mse': 0.0002973641676362604, 'rmse': 0.01724...",2026-02-17 18:20:39.928583
1,2,9b881d69-d956-4f7e-83eb-7380688adb28,iot_pond_3,turbidity,5000,2021-09-15 17:00:00,2021-10-12 14:00:00,LSTM,"[b'P', b'K', b'\x03', b'\x04', b'\x14', b'\x00...","{'scaler_min': -2.5471698113207553, 'scaler_sc...","{'mse': 6.884602044010535e-05, 'rmse': 0.00829...",2026-02-17 18:20:40.506330
2,3,9b881d69-d956-4f7e-83eb-7380688adb28,iot_pond_3,dissolved_oxygen,5000,2021-09-15 17:00:00,2021-10-12 14:00:00,LSTM,"[b'P', b'K', b'\x03', b'\x04', b'\x14', b'\x00...","{'scaler_min': -68.0982367758176, 'scaler_scal...","{'mse': 0.005662249866873026, 'rmse': 0.075247...",2026-02-17 18:20:41.341393
3,4,9b881d69-d956-4f7e-83eb-7380688adb28,iot_pond_3,ph,5000,2021-09-15 17:00:00,2021-10-12 14:00:00,LSTM,"[b'P', b'K', b'\x03', b'\x04', b'\x14', b'\x00...","{'scaler_min': -0.10218897770892146, 'scaler_s...","{'mse': 0.0029574004001915455, 'rmse': 0.05438...",2026-02-17 18:20:52.550343
4,5,9b881d69-d956-4f7e-83eb-7380688adb28,iot_pond_3,ammonia,5000,2021-09-15 17:00:00,2021-10-12 14:00:00,LSTM,"[b'P', b'K', b'\x03', b'\x04', b'\x14', b'\x00...","{'scaler_min': 0.0, 'scaler_scale': 2710.02710...","{'mse': 0.0034602934028953314, 'rmse': 0.05882...",2026-02-17 18:20:54.389534


In [20]:
# Extract the first row
query = f"SELECT model_object, model_weights FROM {table_name} LIMIT 1;"
cursor.execute(query)
row = cursor.fetchone()
row

(<memory at 0x74fa898c19c0>,
 {'scaler_min': -1.0595291013947994, 'scaler_scale': 0.08050538850365678})

In [ ]:
model_binary = row[0]
weights = row[1] 

with tempfile.NamedTemporaryFile(suffix='.keras', delete=False) as tmp:
    tmp.write(bytes(model_binary)) 
    tmp_path = tmp.name

    # Load model
    model = tf.keras.models.load_model(tmp_path)

    previous_data = [[[0.1], [0.2], [0.15] <- 10 values inside]] # Extract the values from sensor-db (last 10 hours)
                                                                # You can find each hour by averaging the values
                                                                #  You can narrow down the data using source_table, target_column	, time_start, time_end
                                                                # values from training_history table.

                                                                # REMEMBER:
                                                                # Scale the data!

    # ------------------------------------------------------------------

    # Example of how to scale the data from Gemini:

    # import numpy as np

    # # Assume raw_data is a list of 10 numbers from your sensor DB
    # # raw_data = [450, 455, 460, 458, 462, 465, 470, 468, 472, 475]

    # # 1. Convert to a NumPy array
    # data_array = np.array(raw_data).reshape(-1, 1)

    # # 2. Apply the scaling weights from your 'weights' dictionary
    # # Scaled = (Raw * Scale) + Min
    # scaled_data = (data_array * weights['scaler_scale']) + weights['scaler_min']

    # # 3. Reshape for LSTM input: (Batch, Window, Features)
    # current_window = scaled_data.reshape(1, 10, 1)

    # print(current_window.shape) # Result: (1, 10, 1)

    # ------------------------------------------------------------------

    # Replace with last 10 hours of the data
    # Data input should already be scaled/
    current_window = previous_data # (Batch Size, Window Size, Features) Shape: (1, 10, 1)
    
    all_real_predictions= []

    for i in range(24): # We loop 24 times to predict 24 hours
        # Run Prediction
        prediction_scaled = model.predict(current_window)
        
        # Revert scaling to get real values
        # Real Value = (Scaled Value - Scaler Min) / Scaler Scale
        real_prediction = (prediction_scaled - weights['scaler_min']) / weights['scaler_scale']
        all_real_predictions.append(real_prediction[0, 0]) # Store this in a list
        print(f"Predicted Value: {real_prediction[0][0]}")

        # Update the window
        #Remove the oldest hour (index 0) and add the new prediction at the end
        new_prediction_reshaped = prediction_scaled.reshape(1, 1, 1) # Used scaled prediction
        current_window = np.append(current_window[:, 1:, :], new_prediction_reshaped, axis=1)

E0000 00:00:1771355192.574433  424375 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1771355192.707905  424375 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model loaded successfully!
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Value: 19.400163650512695


In [ ]:
# Prediction for next 24 hours
print(all_real_predictions)

In [ ]:
cursor.close()
conn.close()